In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('data/student_data_cleaned.csv')

X = pd.get_dummies(df.drop(columns='Exam_Score'), drop_first=True)
y = df['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Features:", X.shape[1])
print("Training rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

baseline = np.full(len(y_test), y_train.mean())
print("\nBaseline (always predict the average)")
print("MAE:", round(mean_absolute_error(y_test, baseline), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, baseline)), 3))

lr = LinearRegression()
lr.fit(X_train, y_train)
pred = lr.predict(X_test)
print("\nLinear Regression")
print("MAE:", round(mean_absolute_error(y_test, pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 3))
print("R2:", round(r2_score(y_test, pred), 3))

Features: 27
Training rows: 5285 | Test rows: 1322

Baseline (always predict the average)
MAE: 2.823
RMSE: 3.761

Linear Regression
MAE: 0.452
RMSE: 1.804
R2: 0.77


In [2]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    'Linear Regression': lr,
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

results = []
for name, model in models.items():
    if name != 'Linear Regression':
        model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'MAE': round(mean_absolute_error(y_test, pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 3),
        'R2': round(r2_score(y_test, pred), 3),
    })

results_df = pd.DataFrame(results).sort_values('MAE')
results_df

,Model,MAE,RMSE,R2
0,Linear Regression,0.452,1.804,0.770
2,Random Forest,1.184,2.233,0.647
1,Decision Tree,1.600,2.511,0.554


In [3]:
from sklearn.model_selection import cross_val_score, KFold
import joblib, os

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    print(f"{name}: average MAE = {scores.mean():.3f} (+/- {scores.std():.3f})")

os.makedirs('models', exist_ok=True)
joblib.dump({'model': lr, 'columns': list(X.columns)}, 'models/best_model.pkl')
print("\nSaved the Linear Regression model to models/best_model.pkl")

Linear Regression: average MAE = 0.487 (+/- 0.070)
Decision Tree: average MAE = 1.660 (+/- 0.082)
Random Forest: average MAE = 1.218 (+/- 0.059)

Saved the Linear Regression model to models/best_model.pkl
